# 01 — Exploratory Data Analysis
This notebook explores the Lending Club dataset before modelling. Run it from the repository root or from this notebook's folder.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = Path('../Data/loan_data.csv') if Path('../Data/loan_data.csv').exists() else Path('Data/loan_data.csv')
df = pd.read_csv(DATA_PATH)
TARGET = 'not.fully.paid'
df.head()

## 1. Dataset structure and quality

In [ ]:
print('Shape:', df.shape)
df.info()
display(df.describe(include='all').T)
display(pd.DataFrame({'missing': df.isna().sum(), 'missing_pct': (df.isna().mean()*100).round(2)}))
print('Duplicate rows:', df.duplicated().sum())

## 2. Target imbalance
Class 1 is the not-fully-paid class and is the main class of interest.

In [ ]:
display(df[TARGET].value_counts().rename('count').to_frame())
display((df[TARGET].value_counts(normalize=True)*100).round(2).rename('percent').to_frame())
sns.countplot(data=df, x=TARGET)
plt.title('Target class distribution')
plt.show()

## 3. Default rates by loan purpose

In [ ]:
purpose_rate = df.groupby('purpose')[TARGET].agg(['mean', 'count']).sort_values('mean', ascending=False)
display(purpose_rate)
purpose_rate['mean'].plot(kind='bar', figsize=(10,4), title='Not-fully-paid rate by purpose', ylabel='Rate')
plt.xticks(rotation=35, ha='right'); plt.tight_layout(); plt.show()

## 4. Numeric variables and correlation
Correlation is a diagnostic, not an automatic feature-removal rule. Retain variables unless there is a clear statistical or domain reason to remove them.

In [ ]:
numeric = df.select_dtypes(include='number')
plt.figure(figsize=(11,8))
sns.heatmap(numeric.corr(), cmap='coolwarm', center=0, annot=True, fmt='.2f')
plt.title('Numeric feature correlation matrix'); plt.tight_layout(); plt.show()

for col in ['int.rate', 'fico', 'dti', 'revol.util', 'log.annual.inc']:
    if col in df.columns:
        sns.boxplot(data=df, x=TARGET, y=col)
        plt.title(f'{col} by target class'); plt.show()

## 5. Initial observations
After running the notebook, record evidence-based observations here: target imbalance, features with visibly different distributions, purpose categories with higher/lower rates, missingness, outliers and limitations. Avoid claiming that correlation proves causation.